In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm
import re

print("Memulai proses sintesis data AI menggunakan model IndoT5")

# Mengecek ketersediaan perangkat komputasi
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Perangkat yang digunakan: {device}")

# Memuat pre-trained tokenizer dan model IndoT5-base-paraphrase
model_repo = "Wikidepia/IndoT5-base-paraphrase"
tokenizer = AutoTokenizer.from_pretrained(model_repo)
model = AutoModelForSeq2SeqLM.from_pretrained(model_repo).to(device)

df_human = pd.read_csv('../data/interim/dataset_human_pre2018_cleaned_tahap1_7_fix.csv')
ai_results = []

print(f"Memproses {len(df_human)} data abstrak untuk sintesis...")

# Mengeksekusi proses parafrase pada setiap baris data
for index, row in tqdm(df_human.iterrows(), total=len(df_human)):
    teks_asli = row['abstract']
    
    # Proses tokenisasi teks masukan
    input_ids = tokenizer.encode(
        teks_asli,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)
    
    # Penyesuaian dinamis batas token keluaran berdasarkan panjang teks input
    input_word_count = len(teks_asli.split())
    target_min = max(50,  int(input_word_count * 0.7))
    target_max = max(150, int(input_word_count * 0.9))
    min_tok = min(int(target_min * 1.3), 200)
    max_tok = min(int(target_max * 1.3), 400)
    
    # Generasi teks sintesis dengan pengaturan parameter beam search
    outputs = model.generate(
        input_ids,
        max_length=max_tok,
        min_length=min_tok,
        num_beams=4,
        no_repeat_ngram_size=3,
        repetition_penalty=1.3,
        length_penalty=1.0,
        early_stopping=True,
    )
    
    ai_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Pascapemrosesan: mengeliminasi klausa yang terlalu pendek (< 5 kata)
    kalimat = ai_text.split('.')
    kalimat_bersih = [k.strip() for k in kalimat if len(k.strip().split()) >= 5]
    ai_text = '. '.join(kalimat_bersih).strip()
    
    if ai_text and not ai_text.endswith('.'):
        ai_text += '.'
    
    ai_results.append({
        'source': row['source'] + " (IndoT5 Parafrase)",
        'year': row['year'],
        'abstract': ai_text,
        'label': 'ai'
    })

# Membentuk DataFrame hasil sintesis
df_ai = pd.DataFrame(ai_results)
df_ai['abstract'] = df_ai['abstract'].apply(lambda x: re.sub(r'\s+', ' ', str(x)).strip())

# Mengekspor dataset kelas AI
df_ai.to_csv('../data/interim/dataset_ai_synthesized_final_9_fix.csv', index=False, encoding='utf-8')

print(f"\nSintesis selesai. {len(df_ai)} data AI tersimpan di dataset_ai_synthesized_final_8_fix.csv")
print("\nMenampilkan 3 sampel hasil sintesis:")
for i, row in df_ai.head(3).iterrows():
    print(f"\nSampel {i+1}: {row['abstract'][:200]}...")

Memulai proses sintesis data AI menggunakan model IndoT5
Perangkat yang digunakan: cuda
Memproses 1138 data abstrak untuk sintesis...


100%|██████████| 1138/1138 [1:14:44<00:00,  3.94s/it]


Sintesis selesai. 1138 data AI tersimpan di 'dataset_ai_synthesized_final_8_fix.csv'

Menampilkan 3 sampel hasil sintesis:

Sampel 1: acar merupakan salah satu komuditi agrobisnis pertanian atau perkebunan di Indonesia yang banyak diminati oleh petani, karena makanan di Indonesia pada umumnya banyak menggunakan Cabai, sehingga merek...

Sampel 2: Cipta Karya, data barang disimpan dalam bentuk tumpukan faktur pembelian yang memungkinkan terjadinya kehilangan data barang, perhitungan setiap transaksi masih menggunakan perhitungan manual sehingga...

Sampel 3: (Technique for Others Preference By Similarity to Ideal Solution (TOPSIS) adalah jawaban atas kesulitan yang dihadapi oleh Kantor Kecamatan Tembakan Kabupaten Inhil dengan menentukan kriteria dan alte...


In [ ]:
import pandas as pd

# Memuat dataset manusia (asli) dan dataset AI (sintesis)
file_manusia = '../data/interim/dataset_human_pre2018_cleaned_tahap1_7_fix.csv'
file_ai = '../data/interim/dataset_ai_synthesized_final_9_fix.csv'

df_human = pd.read_csv(file_manusia)
df_ai = pd.read_csv(file_ai)

# Memastikan kedua dataset memiliki jumlah baris yang sama
print(f"Total data manusia: {len(df_human)} | Total data AI: {len(df_ai)}")

# Memilih indeks sampel untuk dibandingkan (misal indeks 0, 15, dan 50)
# Lu bisa ganti angkanya untuk nyari kalimat perbandingan yang paling bagus buat laporan
indeks_sampel = [0, 15, 50] 

print("\nPerbandingan Teks Manusia vs AI:\n")
for i in indeks_sampel:
    print(f"=== INDEKS {i} ===")
    print(f"[MANUSIA] : {df_human.loc[i, 'abstract']}")
    print(f"[AI]      : {df_ai.loc[i, 'abstract']}\n")

Total data manusia: 1138 | Total data AI: 1138

Perbandingan Teks Manusia vs AI:

=== INDEKS 0 ===
[MANUSIA] : Kemajuan teknologi banyak memberian pengaruh dalam proses pekerjaan dibidang pertanian, dimana banyak peralatan pertanian yang dikembangkan sehingga proses pekerjaan pertanian dapat diselesaikan dengan baik dan memberikan hasil yang lebih baik dari segi kualitas maupun kuantitas. Cabai merupakan salah satu komuditi agrobisnis pertanian atau perkebunan di indonesia yang banyak diminati oleh para petani. Hal ini disebabkan makanan di Indonesia pada umumnya banyak menggunakan cabai, sehingga cabai menjadi komuditi yang sangat menjanjikan bagi petani. Masalahnya pada saat tanaman cabai terinfeksi oleh penyakit maupun hama tertentu, sehingga para petani perlu untuk mendiagnosa tanamannya tersebut, maka bukan saja akan membuat biaya untuk pembelian pestisida yang membengkak akan tetapi juga dapat mengakibatkan tanaman cabai mati sehingga panen menjadi gagal. Untuk itulah perlu dikem

In [2]:
import pandas as pd

# ── LOAD DATASET TERPISAH SEBELUM TRUNCATE ───────────────────────────
# Ganti nama file di bawah ini sesuai dengan nama file CSV lu di folder
file_human = '../data/interim/dataset_human_pre2018_cleaned_tahap1_7_fix.csv' 
file_ai = '../data/interim/dataset_ai_synthesized_final_9_fix.csv'

df_human = pd.read_csv(file_human)
df_ai = pd.read_csv(file_ai)

# ── AMBIL TEKS PADA INDEX PERTAMA (0) ───────────────────────────────
teks_manusia = str(df_human.loc[0, 'abstract'])
teks_ai = str(df_ai.loc[0, 'abstract'])

# Menghitung jumlah kata (split berdasarkan spasi)
jml_kata_manusia = len(teks_manusia.split())
jml_kata_ai = len(teks_ai.split())

# ── TAMPILKAN HASIL ─────────────────────────────────────────────────
print("="*70)
print("ANALISIS JUMLAH KATA INDEX 0 (SEBELUM PREPROCESSING TAHAP 2)")
print("="*70)

print("\n[1] DATA MANUSIA (Hasil Prep 1)")
print(f"Jumlah Kata : {jml_kata_manusia} kata")
print(f"Cuplikan    :\n{teks_manusia[:300]}...\n")

print("-" * 70)

print("\n[2] DATA AI (Hasil Sintesis IndoT5)")
print(f"Jumlah Kata : {jml_kata_ai} kata")
print(f"Cuplikan    :\n{teks_ai[:300]}...\n")

print("="*70)
# Menghitung selisih penyusutan kata
penyusutan = jml_kata_manusia - jml_kata_ai
print(f"KESIMPULAN: Terjadi pemangkasan/penyusutan sebanyak {penyusutan} kata oleh model IndoT5.")
print("="*70)

ANALISIS JUMLAH KATA INDEX 0 (SEBELUM PREPROCESSING TAHAP 2)

[1] DATA MANUSIA (Hasil Prep 1)
Jumlah Kata : 182 kata
Cuplikan    :
Kemajuan teknologi banyak memberian pengaruh dalam proses pekerjaan dibidang pertanian, dimana banyak peralatan pertanian yang dikembangkan sehingga proses pekerjaan pertanian dapat diselesaikan dengan baik dan memberikan hasil yang lebih baik dari segi kualitas maupun kuantitas. Cabai merupakan sal...

----------------------------------------------------------------------

[2] DATA AI (Hasil Sintesis IndoT5)
Jumlah Kata : 125 kata
Cuplikan    :
acar merupakan salah satu komuditi agrobisnis pertanian atau perkebunan di Indonesia yang banyak diminati oleh petani, karena makanan di Indonesia pada umumnya banyak menggunakan Cabai, sehingga mereka dapat menggantikan peran seorang ahli dalam mencari solusi dan membuat panen menjadi gagal. Sistem...

KESIMPULAN: Terjadi pemangkasan/penyusutan sebanyak 57 kata oleh model IndoT5.
